# Training an EAGLE3 Draft Head for Cosmos-Reason2

This notebook walks through the full workflow for training an EAGLE3 speculative-decoding draft head on top of [nvidia/Cosmos-Reason2-8B](https://huggingface.co/nvidia/Cosmos-Reason2-8B).

**Workflow overview**

| Step | Description |
| :---: | :--- |
| 1 | Install dependencies |
| 2 | Authenticate with Hugging Face |
| 3 | Prepare training data from the Nemotron dataset |
| 4 | Calibrate the draft vocabulary |
| 5 | Launch training |
| 6 | Export checkpoint for deployment |

> **Hardware requirement** – Cosmos-Reason2-8B requires at least one 80 GB GPU (e.g. H100/A100).
> Multi-GPU training is supported automatically via FSDP2 when more than one GPU is available.

## Step 1 – Install Dependencies

In [ ]:
%%bash
pip install -U nvidia-modelopt[hf]
pip install -r ../requirements.txt

## Step 2 – Authenticate with Hugging Face

Both `nvidia/Cosmos-Reason2-8B` and `nvidia/Nemotron-Post-Training-Dataset-v2` require accepting
their licence agreements on the Hub. Run the cell below and follow the interactive prompt to log in:

In [ ]:
%%bash
hf auth login

## Step 3 – Prepare Training Data

We use a curated subset of [nvidia/Nemotron-Post-Training-Dataset-v2](https://huggingface.co/datasets/nvidia/Nemotron-Post-Training-Dataset-v2)
(chat split) for training.  The `nemotron_mapping.bin` file (bundled alongside this notebook) selects the specific rows to use.
It stores 0-based dataset row indices as packed `int32` values (little-endian, produced by `numpy.ndarray.tofile`).

The script streams only the required parquet shards and writes a conversation file in the
standard `jsonl` format expected by `launch_train.sh`.

In [ ]:
%%bash
python ../prepare_input_conversations/add_nemotron_chat.py \
    --mapping-file nemotron_mapping.bin

In [ ]:
%%bash
# Expect exactly 89511 conversations.
count=$(wc -l < input_conversations/nemotron-chat.jsonl)
echo "${count} conversations in ../input_conversations/nemotron-chat.jsonl"
[ "$count" -eq 89511 ] || { echo "ERROR: expected 89511, got ${count}"; exit 1; }

## Step 4 – Calibrate the Draft Vocabulary

`CR2_eagle_config.json` sets `"draft_vocab_size": 32000`.  Using a compressed vocabulary
speeds up training and inference, but requires a one-time calibration step that produces a
token-mapping file (`d2t.pt`).

In [ ]:
%%bash
python ../scripts/calibrate_draft_vocab.py \
    --model nvidia/Cosmos-Reason2-8B \
    --data input_conversations/nemotron-chat.jsonl \
    --draft_vocab_size 32000 \
    --save_dir draft_vocab_cache

## Step 5 – Train the EAGLE3 Draft Head

Training is launched via `launch_train.sh`, which internally calls `accelerate launch main.py`
and sets up FSDP2 automatically when multiple GPUs are available.

Key arguments used for Cosmos-Reason2:

| Argument | Value | Notes |
| :--- | :--- | :--- |
| `--model` | `nvidia/Cosmos-Reason2-8B` | Target VLM |
| `--data` | `guides/input_conversations/nemotron-chat.jsonl` | Training conversations |
| `--eagle_config` | `guides/CR2_eagle_config.json` | Draft-head architecture |
| `--draft_vocab_cache` | `guides/draft_vocab_cache/Cosmos-Reason2-8B/d2t.pt` | Token-mapping from Step 4 |
| `--vlm_processor` | `nvidia/Cosmos-Reason2-8B` | VLM image processor |
| `--vlm_img_dir` | `data/` | Directory containing referenced images |
| `--training_seq_len` | `16384` | Max token length per sample (lower to save GPU memory or speed up training) |
| `--lr` | `1.5e-4` | Learning rate |
| `--num_epochs` | `20` | Training epochs |
| `--train_bs` | `1` | Per-device batch size |
| `--save_steps` | `1000` | Checkpoint frequency |
| `--ar_validate_steps` | `1000000` | Effectively disables in-training AR validation |

> **Tip** – Set `--ar_validate_steps` to a smaller value (e.g. `500`) to periodically measure
> acceptance rate on MT-Bench during training.

In [ ]:
%%bash
export WANDB_MODE=disabled
OUTPUT_DIR=ckpts/cosmos-reason2-8b-eagle3
EAGLE_CONFIG=guides/CR2_eagle_config.json
DRAFT_VOCAB_CACHE=guides/draft_vocab_cache/Cosmos-Reason2-8B/d2t.pt


# 20 epochs on 89k samples (4xB100): ~24 hours.
cd ..; OUTPUT_DIR=$OUTPUT_DIR ./launch_train.sh \
  --model nvidia/Cosmos-Reason2-8B \
  --output_dir $OUTPUT_DIR \
  --data guides/input_conversations/nemotron-chat.jsonl \
  --lr 1.5e-4 \
  --num_epochs 20 \
  --train_bs 1 \
  --eagle_config $EAGLE_CONFIG \
  --draft_vocab_cache $DRAFT_VOCAB_CACHE \
  --training_seq_len 16384 \
  --save_steps 1000 \
  --ar_validate_steps 1000000 \
  --vlm_processor nvidia/Cosmos-Reason2-8B \
  --vlm_img_dir data/

## Step 6 – Export Checkpoint for Deployment

After training completes, convert the ModelOpt checkpoint to the Hugging Face–compatible
format expected by vLLM.  Point `--model_path` to the desired checkpoint subdirectory
(e.g. `checkpoint-110000`).

In [ ]:
%%bash
CKPT_DIR=ckpts/cosmos-reason2-8b-eagle3/checkpoint-110000
EXPORT_PATH=export/cosmos-reason2-8b-eagle3

python scripts/export_hf_checkpoint.py \
    --model_path $CKPT_DIR \
    --export_path $EXPORT_PATH

## Deployment

The exported checkpoint can be served directly with **vLLM**:

```bash
vllm serve nvidia/Cosmos-Reason2-8B \
    --host 0.0.0.0 \
    --port 8000 \
    --speculative_config '{"method": "eagle3", "model": "export/cosmos-reason2-8b-eagle3", "num_speculative_tokens": 3}'
```

Refer to the [vLLM speculative decoding docs](https://docs.vllm.ai/en/latest/features/spec_decode/) for the full list of options.